# 06 · Energy landscape visualization

Estimate and plot the pseudo-energy surface $E(z) = -\log p(z)$ over the latent cloud, with one participant's trajectory overlaid.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt


In [ ]:
from data.synthetic_generator import generate_synthetic_cohort
from features import engineer_all_features
from states.latent_state_encoder import encode_latent_states_classical
from states.state_geometry import project_latent_2d
from states.energy_landscape import estimate_energy_landscape, plot_energy_landscape

df = engineer_all_features(generate_synthetic_cohort(n_participants=60, n_days=120, seed=17))


In [ ]:
def pick(df, cols):
    present = [c for c in cols if c in df.columns]
    return df[present].to_numpy(dtype=float, na_value=0.0) if present else np.zeros((len(df), len(cols)))

W = pick(df, ['sleep_duration_hours','hrv_rmssd','resting_hr','daily_steps','recovery_score'])
B = pick(df, ['screen_time_minutes','mobility_radius_km','location_entropy','phone_unlock_count'])
C = pick(df, ['temperature_c','nighttime_temperature_c','aqi','heat_wave_flag'])
M = pick(df, ['missing_wearable_flag','missing_phone_flag','missing_survey_flag'])
P = pick(df, ['baseline_hrv','baseline_resting_hr','baseline_climate_vulnerability','baseline_resilience'])
Z = encode_latent_states_classical(W, B, C, M, P).latent

landscape = estimate_energy_landscape(Z, grid_size=60)
fig, ax = plt.subplots(figsize=(7, 6))
plot_energy_landscape(landscape, ax=ax)
pid = df['participant_id'].iloc[0]
pZ = Z[df['participant_id'].values == pid]
pZ2 = project_latent_2d(pZ)
ax.plot(pZ2[:,0], pZ2[:,1], color='white', lw=1.2)
plt.show()
